In [2]:
import pandas as pd
from scipy.stats import randint
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, roc_auc_score
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

In [6]:
RANDOM_STATE = 42

df = pd.read_csv(r"C:\Users\Lenovo\Downloads\customer_churn_tuning_data.csv")
X = df.drop(columns=["churn"])
y = df["churn"]

df


,age,monthly_spend_usd,tenure_months,support_tickets,avg_session_mins,login_frequency_weekly,churn
0,23,89.78,55,2,1.1,6.3,0
1,18,30.90,15,2,22.9,6.4,0
2,31,53.00,36,0,12.2,3.9,0
3,24,95.07,56,0,1.0,5.1,0
4,37,25.51,41,0,20.5,0.1,0
...,...,...,...,...,...,...,...
295,26,62.14,1,0,20.6,3.6,0
296,39,10.00,53,1,30.7,0.0,0
297,24,65.78,42,2,10.5,4.6,1
298,18,48.98,31,0,10.2,7.8,0


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [7]:
searches = {
    "SVC (grid)": GridSearchCV(
        estimator=Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(probability=True, random_state=RANDOM_STATE)),
        ]),
        param_grid={
            "clf__C": [0.1, 1, 10, 50],
            "clf__kernel": ["linear", "rbf"],
            "clf__gamma": ["scale", "auto", 0.01, 0.1],
        },
        cv=cv,
        scoring="f1",
        n_jobs=-1,
    ),
    "LogisticRegression (grid)": GridSearchCV(
        estimator=Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]),
        param_grid={
            "clf__C": [0.01, 0.1, 1, 10, 100],
            "clf__solver": ["lbfgs", "liblinear"],
        },
        cv=cv,
        scoring="f1",
        n_jobs=-1,
    ),
    "RandomForest (random)": RandomizedSearchCV(
        estimator=RandomForestClassifier(random_state=RANDOM_STATE),
        param_distributions={
            "n_estimators": randint(50, 400),
            "max_depth": [None, 3, 5, 8, 12],
            "min_samples_split": randint(2, 12),
            "min_samples_leaf": randint(1, 6),
            "max_features": ["sqrt", "log2", None],
        },
        n_iter=30,
        cv=cv,
        scoring="f1",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
}

In [8]:
results = []
for name, search in searches.items():
    search.fit(X_train, y_train)
    results.append({
        "model": name,
        "cv_f1": round(search.best_score_, 4),
        "best_params": search.best_params_,
    })

summary_df = pd.DataFrame(results).sort_values(by="cv_f1", ascending=False).reset_index(drop=True)
print(summary_df.to_string(index=False))

C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


                    model  cv_f1                                                                                               best_params
               SVC (grid) 0.7955                                                   {'clf__C': 50, 'clf__gamma': 0.1, 'clf__kernel': 'rbf'}
    RandomForest (random) 0.7847 {'max_depth': 8, 'max_features': None, 'min_samples_leaf': 4, 'min_samples_split': 9, 'n_estimators': 84}
LogisticRegression (grid) 0.6858                                                                   {'clf__C': 100, 'clf__solver': 'lbfgs'}


In [9]:
best_name = summary_df.loc[0, "model"]
best_model = searches[best_name].best_estimator_

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print(f"\nBest model: {best_name}")
print(f"Test F1 score:  {f1_score(y_test, y_pred):.4f}")
print(f"Test ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}\n")
print(classification_report(y_test, y_pred, target_names=["Retained (0)", "Churned (1)"]))


Best model: SVC (grid)
Test F1 score:  0.8095
Test ROC-AUC:   0.8840

              precision    recall  f1-score   support

Retained (0)       0.90      0.90      0.90        39
 Churned (1)       0.81      0.81      0.81        21

    accuracy                           0.87        60
   macro avg       0.85      0.85      0.85        60
weighted avg       0.87      0.87      0.87        60

